In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torch-snippets==0.499.0
#!pip install torchsummary 

In [ ]:
pip install --upgrade torch_snippets


In [ ]:
from torch_snippets import *
from torchvision import transforms
from sklearn.model_selection import train_test_split
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import cv2 
import numpy as np 
import pandas as pd 
import nibabel as nib
import glob 

In [ ]:
image_paths=sorted(glob.glob('/kaggle/input/people-clothing-segmentation/png_images/IMAGES/*.png'))
mask_paths=sorted(glob.glob('/kaggle/input/people-clothing-segmentation/png_masks/MASKS/*.png'))

In [ ]:
train_images=image_paths[:int(0.8*(len(image_paths)))]
train_labels=mask_paths[:int(0.8*(len(mask_paths)))]
test_images=image_paths[int(0.8*(len(image_paths))):]
test_label=mask_paths[int(0.8*(len(mask_paths))):]

In [ ]:
tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # imagenet
])

In [ ]:
class SegData(Dataset):
    def __init__(self,images,labels):
        self.images=images
        self.labels=labels
    def __len__(self):
        return len(self.images)
    def __getitem__(self, ix):
        image = cv2.imread(self.images[ix])
        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (512,512))
        mask = read(self.labels[ix],1)
        mask = cv2.resize(mask, (512,512))
        mask=cv2.cvtColor(mask,cv2.COLOR_RGB2GRAY)
        return image, mask
    def choose(self): return self[randint(len(self))]
    def collate_fn(self, batch):
        ims, masks = list(zip(*batch))
        ims = torch.cat([tfms(im.copy()/255.)[None] for im in ims]).float().to(device)
        ce_masks = torch.cat([torch.Tensor(mask)[None] for mask in masks]).long().to(device)
        return ims, ce_masks

In [ ]:
train_dataset=SegData(train_images,train_labels)
test_dataset=SegData(test_images,test_label)

In [ ]:
image,mask=train_dataset[10]
print(image.shape)
print(mask.shape)
plt.imshow(image)
plt.figure()
plt.imshow(mask)
print(image.shape)
print(mask.max())

In [ ]:
from torch import nn 
import torchvision
import torch.nn.functional as F

In [ ]:
def conv(nin,nout,kernel_size):
  return nn.Sequential(
          nn.Conv2d(nin,nout, kernel_size=kernel_size,padding="same"),
          nn.BatchNorm2d(nout),
          nn.ReLU(inplace=True))
  
class resconv(nn.Module):
  def __init__(self,in_features,out_features):
    super(resconv,self).__init__()
    self.block=nn.Sequential(
        nn.ReflectionPad2d(1),
        nn.Conv2d(in_features,out_features,3),
        nn.InstanceNorm2d(out_features),
        nn.ReLU(inplace=True),
        nn.ReflectionPad2d(1),
        nn.Conv2d(out_features,out_features,3),
        nn.InstanceNorm2d(out_features),
        nn.ReLU(inplace=True),
        
    )
  def forward(self,x):
      return x+self.block(x)

def up_conv(nin,nout):
  return nn.Sequential(
      nn.ConvTranspose2d(nin,nout,kernel_size=2,stride=2),
      nn.InstanceNorm2d(nout),
      nn.ReLU(),
  )

In [ ]:
class ResUnet1(nn.Module):
  def __init__(self):
    super().__init__()
    self.block1=conv(3,64,(7,7))
    self.max_pool=nn.MaxPool2d(2,stride=2)
    self.block2=conv(64,128,(1,1))
    self.block3=conv(128,256,(1,1))
    self.block4=conv(256,512,(1,1))
    self.block2_1=resconv(64,64)
    self.block3_1=resconv(128,128)
    self.block4_1=resconv(256,256)
    self.block_5_1=resconv(512,512)
    self.upconv5=up_conv(512,512)
    self.upconv4=up_conv(512,256)
    self.upconv3=up_conv(256,128)
    self.upconv2=up_conv(128,64)
    self.conv6=conv(512+512,512,kernel_size=(3,3))
    self.conv5=conv(256+256,256,kernel_size=(3,3))
    self.conv4=conv(128+128,128,kernel_size=(3,3))
    self.conv3=conv(64+64,64,kernel_size=(3,3))
    self.last_conv=nn.Conv2d(64,59,kernel_size=(1,1),padding="same")
  def forward(self,x):
    x1=self.block1(x)
    x2=self.max_pool(x1)
    x3=self.block2_1(x2)
    x4=self.block2(x3)
    x5=self.max_pool(x4)
    x6=self.block3_1(x5)
    x7=self.block3(x6)
    x8=self.max_pool(x7)
    x9=self.block4_1(x8)
    x10=self.block4(x9)
    x11=self.max_pool(x10)
    x12=self.block_5_1(x11)
    y12=self.upconv5(x12)
    y_out1=torch.cat([y12,x10],dim=1)
    y_out1=self.conv6(y_out1)
    y11=self.upconv4(y_out1)
    y_out2=torch.cat([y11,x7],dim=1)
    y_out2=self.conv5(y_out2)
    y10=self.upconv3(y_out2)
    y_out3=torch.cat([y10,x4],dim=1)
    y_out3=self.conv4(y_out3)
    y9=self.upconv2(y_out3)
    y_out=torch.cat([y9,x1],dim=1)
    y_out=self.conv3(y_out)
    out=self.last_conv(y_out)
    return out


In [ ]:
model=ResUnet1()
model=model.to(device)


In [ ]:
ce = nn.CrossEntropyLoss()
def UnetLoss(preds, targets):
    ce_loss = ce(preds, targets)
    acc = (torch.argmax(preds,1) == targets).float().mean()
    return ce_loss, acc

In [ ]:
criterion=UnetLoss
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
scheduler=torch.optim.lr_scheduler.StepLR(optimizer,gamma=0.5,step_size=15)

In [ ]:
train_dataloader=torch.utils.data.DataLoader(train_dataset,batch_size=8,shuffle=True,collate_fn=train_dataset.collate_fn)
test_dataloader=torch.utils.data.DataLoader(test_dataset,batch_size=8,shuffle=False,collate_fn=test_dataset.collate_fn)

In [ ]:
image,mask=next(iter(train_dataloader))
image.shape

In [ ]:
def train_batch(model, data, optimizer, criterion):
    ims, ce_masks = data
    ims=ims
    ce_masks=ce_masks
    model.train()
    _masks = model(ims)
    optimizer.zero_grad()
    loss, acc = criterion(_masks, ce_masks)
    loss.backward()
    optimizer.step()
    return loss.item(), acc.item()
@torch.no_grad()
def valid_batch(model, data, criterion):
    model.eval()
    ims, ce_masks = data
    ims=ims
    ce_masks=ce_masks
    _masks = model(ims)
    loss, acc = criterion(_masks, ce_masks)
    return loss.item(), acc.item()

In [ ]:
n_epochs=5
log = Report(n_epochs)
patience=0
p_threshold=5
val_loss=[]
for ex in range(n_epochs):
    N = len(train_dataloader)
    for bx, data in enumerate(train_dataloader):
        loss, acc = train_batch(model, data, optimizer, criterion)
        log.record(ex+(bx+1)/N, trn_loss=loss, trn_acc=acc, end='\r')
    scheduler.step()
    N = len(test_dataloader)
    for bx, data in enumerate(test_dataloader):
        loss, acc = valid_batch(model, data,criterion)
        log.record(ex+(bx+1)/N, val_loss=loss, val_acc=acc, end='\r')
        val_loss.append(loss)
    log.report_avgs(ex+1)
    if (log.history('val_loss')[-1]-log.history('val_loss')[-2])>1:
        patience+=1
    if patience>p_threshold:
        break 
    

In [ ]:
test_dataloader=torch.utils.data.DataLoader(test_dataset,batch_size=8,shuffle=True,collate_fn=test_dataset.collate_fn)
with torch.no_grad():
    model.eval()
    img,mask=next(iter(test_dataloader))
    _mask=model(img)
    _,_mask_m=torch.max(_mask,dim=1)
    print(_mask.shape)
    plt.imshow(img[0].detach().cpu().permute(1,2,0).numpy())
    plt.figure()
    plt.imshow(_mask_m[0].detach().cpu().numpy(),cmap='gray')
    plt.figure()
    plt.imshow(mask[0].detach().cpu().numpy(),cmap='gray')

# !!적용!!

In [ ]:
example_paths = sorted(glob.glob('/kaggle/input/example-img/test_img.jpg'))
image_path = example_paths[0]
image_path

In [ ]:
example_img = Image.open(image_path).convert('RGB')
example_img

In [ ]:
example_img.size

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torchvision.transforms as transforms


tfms = transforms.Compose([
    transforms.Resize((512, 512)),  # 이미지 크기를 조정합니다.
    transforms.ToTensor(),  # 이미지를 텐서로 변환합니다.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 이미지를 정규화합니다.
])

# 이미지에 전처리를 적용합니다.
example_img_tensor = tfms(example_img).unsqueeze(0)  # 배치 차원을 추가합니다.

# GPU를 사용할 경우
if torch.cuda.is_available():
    example_img_tensor = example_img_tensor.cuda()

# 모델에 이미지를 전달하여 segmentation을 수행합니다.
with torch.no_grad():
    model.eval()
    output = model(example_img_tensor)
    print("Output shape:", output.shape)  # 출력된 텐서의 크기 출력


num_classes = 59
colors = [[random.randint(0, 511) for _ in range(3)] for _ in range(num_classes)]


# 세그멘테이션 맵을 이미지로 변환
segmentation_map = output.argmax(dim=1).cpu().numpy()  # 가장 높은 확률을 가진 클래스의 인덱스 선택


def colorize_segmentation_map(segmentation_map, colors):
    # 세그멘테이션 맵을 RGB 이미지로 변환
    num_classes = len(colors)
    height, width = segmentation_map.shape[1:]
    colored_map = np.zeros((height, width, 3), dtype=np.uint8)
    for i in range(height):
        for j in range(width):
            class_idx = segmentation_map[0, i, j]  # 각 픽셀의 클래스 인덱스
            if class_idx < num_classes:  # 클래스 인덱스가 유효한 경우에만 색상 지정
                colored_map[i, j] = colors[class_idx]  # 색상 지정
    return colored_map

# 색상을 적용한 이미지 생성
colored_image = colorize_segmentation_map(segmentation_map, colors)

# 이미지 시각화
plt.imshow(colored_image)

In [ ]:
gray_image = np.squeeze(segmentation_map) * (255 // (num_classes - 1))

# 이미지 시각화
plt.imshow(gray_image, cmap='gray')

In [ ]:
gray_image = np.squeeze(segmentation_map) * (255 // (num_classes - 1))

# 이미지 시각화
plt.imshow(gray_image, cmap='gray')

In [ ]:
tfms = transforms.Compose([
    transforms.RandomResizedCrop((512, 512), scale=(0.5, 1.0)),  # 무작위 자르기 및 크기 조정
    transforms.RandomHorizontalFlip(),  # 무작위 수평 뒤집기
    transforms.RandomRotation(15),  # 무작위 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # 색상 변화
    transforms.ToTensor(),  # 이미지를 텐서로 변환
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 이미지를 정규화
])


# 이미지에 전처리를 적용합니다.
example_img_tensor = tfms(example_img).unsqueeze(0)  # 배치 차원을 추가합니다.

# GPU를 사용할 경우
if torch.cuda.is_available():
    example_img_tensor = example_img_tensor.cuda()

# 모델에 이미지를 전달하여 segmentation을 수행합니다.
with torch.no_grad():
    model.eval()
    output = model(example_img_tensor)
#     print("Output shape:", output.shape)  # 출력된 텐서의 크기 출력


num_classes = 59
colors = [[random.randint(0, 511) for _ in range(3)] for _ in range(num_classes)]


# 세그멘테이션 맵을 이미지로 변환
segmentation_map = output.argmax(dim=1).cpu().numpy()  # 가장 높은 확률을 가진 클래스의 인덱스 선택

# 색상을 적용한 이미지 생성
colored_image = colorize_segmentation_map(segmentation_map, colors)

# 이미지 시각화
plt.imshow(colored_image)

In [ ]:
tfms = transforms.Compose([
    transforms.RandomResizedCrop((512, 512), scale=(0.5, 1.0)),  # 무작위 자르기 및 크기 조정
    transforms.RandomHorizontalFlip(),  # 무작위 수평 뒤집기
    transforms.RandomRotation(15),  # 무작위 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # 색상 변화
    transforms.ToTensor(),  # 이미지를 텐서로 변환
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 이미지를 정규화
])


# 이미지에 전처리를 적용합니다.
example_img_tensor = tfms(example_img).unsqueeze(0)  # 배치 차원을 추가합니다.

# GPU를 사용할 경우
if torch.cuda.is_available():
    example_img_tensor = example_img_tensor.cuda()

# 모델에 이미지를 전달하여 segmentation을 수행합니다.
with torch.no_grad():
    model.eval()
    output = model(example_img_tensor)
#     print("Output shape:", output.shape)  # 출력된 텐서의 크기 출력


num_classes = 59
colors = [[random.randint(0, 511) for _ in range(3)] for _ in range(num_classes)]


# 세그멘테이션 맵을 이미지로 변환
segmentation_map = output.argmax(dim=1).cpu().numpy()  # 가장 높은 확률을 가진 클래스의 인덱스 선택

# 색상을 적용한 이미지 생성
colored_image = colorize_segmentation_map(segmentation_map, colors)

# 이미지 시각화
plt.imshow(colored_image)

In [ ]:
for class_idx in range(59):
    class_prob = output[0, class_idx].detach().cpu().numpy()
    print(f"Class {class_idx}: Probability = {class_prob}")